In [2]:
import numpy as np
import pandas as pd
import yfinance as yf

# Mean-Variance Portfolio Optimization and Efficient Frontier Analysis
---

## 1. Introduction

Portfolio optimization is a fundamental problem in quantative finance.
The goal is to allocate capital across multiple assets in a way that optimizes the tradeoff between expected return and risk.

In this project, we study a portfolio composed of 8 US stocks.
Although the assets are denominated in USD, performance will be evaluated from the perspective of a EUR-based investor.

We allow short-selling and include a risk-free asset in order to construct:

- The Global Minimum Variance Portfolio
- The Efficient Frontier
- The Maximum Sharpe (Tangency) Portfolio

The results will be compared against a simple equal-weight benchmark portfolio.


## 2. Research question

This project investigates how mean-variance portfolio optimization with short-selling performs relative to an equal-weight portfolio for a set of 8 US stocks, when evaluated of the perspective of a EUR-based investor including risk-free asset.

## 3. Assets

The portfolio consists of 8 large-cap US stocks across different sectors:

- Technology: AAPL, MSFT
- Healthcare: JNJ, PFE
- Consumer: KO, WMT
- Energy: XOM
- Finance: JPM

This diversified sector exposure allows meaningful covariance structure analysis and realistic portfolio construction.

## 4. Data description

We use 5 years of historical daily price data for the selected 8 US stocks

- Frequency: Daily
- Data type: Adjusted Close prices
- Period: Last 5 years
- Currency of assets: USD
- Base evaluation currency: EUR

Daily returns will be computed from adjusted closing prices.
The analysis will use log-returns for mathematical consistency.


## 5. Data download and currency conversion

We download 5 years of daily adjusted closing prices for the selected US stocks.

Since the assets are denominated in USD but evaluated from a EUR-based investor perspective, we convert prices into EUR using the EUR/USD exchange rate.


In [10]:
# Defining tickers
tickers = ["AAPL", "MSFT", "JNJ", "PFE", "KO", "WMT", "XOM", "JPM"]
fx_ticker = "EURUSD=X"

# Defining 5 years date range
end_date = pd.Timestamp.today().normalize()
start_date = end_date - pd.DateOffset(years=5)

# Downloading stock prices in USD and extracting close prices
stocks_raw = yf.download(tickers, start=start_date, end=end_date, progress=False)
prices_usd = stocks_raw['Close'].dropna(how='all')

# Downloading EUR/USD rate
fx_raw = yf.download(fx_ticker, start=start_date, end=end_date, progress=False)
eurusd = fx_raw[("Close", fx_ticker)].dropna()

# Converting USD prices to EUR
eur_per_usd = 1 / eurusd
data = prices_usd.join(eur_per_usd.rename("EUR_per_USD"), how="inner")
prices_eur = data[tickers].mul(data["EUR_per_USD"], axis=0).dropna()

prices_eur.head()


,AAPL,MSFT,JNJ,PFE,KO,WMT,XOM,JPM
Date,,,,,,,,
2021-03-02,101.124683,186.217230,114.587225,21.662373,35.767230,33.624135,38.416617,109.618908
2021-03-03,98.370964,180.677653,112.249461,22.168023,35.580080,32.879114,38.614802,111.420266
2021-03-04,97.095157,180.538587,110.303726,22.109222,35.654292,32.958567,40.226876,110.024790
2021-03-05,98.817819,185.694276,113.266630,22.386100,36.512398,33.600713,42.037252,111.044724
2021-03-08,95.034055,182.962445,114.613140,22.439004,37.254509,33.395520,42.144126,112.913600


## 6. Return calculation

Portfolio operates on asset returns rather than prices.

We compute daily-log returns defined as:
$$ r_t =  \ln\left(\frac{P_t}{P_{t - 1}}\right) $$

where:

- $ P_t \text{ is the asset price at time } t $
- $ P_{t-1} \text{ is at previous time step}$

Log returns are preferred because:

- They are time additive
- They simplify mathematical derivations
- They are commonly used in quantative finance





In [11]:
log_returns = np.log(prices_eur / prices_eur.shift(1))
log_returns = log_returns.dropna()
log_returns.head()

,AAPL,MSFT,JNJ,PFE,KO,WMT,XOM,JPM
Date,,,,,,,,
2021-03-03,-0.027609,-0.030199,-0.020613,0.023074,-0.005246,-0.022406,0.005146,0.016299
2021-03-04,-0.013054,-0.000770,-0.017486,-0.002656,0.002084,0.002414,0.040900,-0.012604
2021-03-05,0.017586,0.028157,0.026507,0.012445,0.023782,0.019296,0.044021,0.009227
2021-03-08,-0.039043,-0.014821,0.011818,0.002360,0.020121,-0.006126,0.002539,0.016690
2021-03-09,0.046063,0.033932,0.008122,0.009125,-0.009002,0.014085,-0.009346,-0.000870
